# Variant evaluation: individual and pairwise views

This notebook presents Variant 0, Variant 1, and Variant 3 individually. Comparisons are strictly pairwise: Variant 0 versus Variant 1, and Variant 1 versus Variant 3.

Phase 1 reports schema/KB linking F1. Phase 2 separates Query structural agreement from Management operation contracts. Phase 3 has no independently scored artifact. Phase 4 uses execution correction for Query tasks and validation correction for Management tasks. Final accuracy remains an overall metric outside the phases.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from variant_comparison_metrics import build_task_aware_evaluation

OUTPUT_DIR = Path('results/eda2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
task_df, phase_summary = build_task_aware_evaluation()
VARIANTS = ['Variant 0', 'Variant 1', 'Variant 3']
COLOURS = {'Variant 0': '#6B7280', 'Variant 1': '#4C78A8', 'Variant 3': '#59A14F'}
TASK_ORDER = ['Query', 'Management']
DIFFICULTY_ORDER = ['easy', 'non_nested_complex', 'nested_complex']
assert task_df.groupby('variant')['instance_id'].nunique().eq(75).all()
phase_summary.set_index('variant')[['overall_accuracy']]

In [ ]:
PRE = {
    'pre_table_f1': 'Table F1', 'pre_column_f1': 'Column F1',
    'pre_join_f1': 'Join-path F1', 'pre_kb_f1': 'KB F1',
}
PLAN = {
    'query_plan_structure_f1': 'Query structure F1',
    'query_plan_structure_exact': 'Query exact',
    'management_operation_exact': 'Mgmt operation',
    'management_target_f1': 'Mgmt target F1',
    'management_predicate_f1': 'Mgmt predicate F1',
    'management_mutation_f1': 'Mgmt mutation F1',
}
POST = {
    'query_initial_execution_success': 'Query initial success',
    'query_retry_rate_after_error': 'Query retry',
    'query_error_resolution_rate': 'Query resolved',
    'management_validation_rejection_rate': 'Mgmt rejected',
    'management_revision_rate_after_rejection': 'Mgmt revised',
    'management_validation_resolution_rate': 'Mgmt resolved',
}

def finish_rate_axis(ax, title):
    ax.set_title(title); ax.set_ylim(0, 1.12); ax.grid(axis='y', alpha=.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))

def label_bars(ax, bars):
    for bar in bars:
        value = bar.get_height()
        if pd.notna(value):
            ax.text(bar.get_x() + bar.get_width()/2, value + .015,
                    f'{value:.0%}', ha='center', fontsize=8)

def plot_metric_set(ax, rows, metrics, variants, title):
    x = np.arange(len(metrics)); width = .72 / len(variants)
    for i, variant in enumerate(variants):
        values = rows.loc[variant, list(metrics)].astype(float).values
        bars = ax.bar(x + (i - (len(variants)-1)/2) * width, values, width,
                      color=COLOURS[variant], label=variant)
        label_bars(ax, bars)
    ax.set_xticks(x); ax.set_xticklabels(metrics.values(), rotation=35, ha='right')
    finish_rate_axis(ax, title)

def accuracy_tables(frame):
    overall = frame['final_passed'].mean()
    by_task = (frame.groupby('category')['final_passed'].mean()
               .reindex(TASK_ORDER))
    by_structure = (frame.groupby(['category', 'structural_difficulty'])['final_passed']
                    .mean().reindex(pd.MultiIndex.from_product(
                        [TASK_ORDER, DIFFICULTY_ORDER],
                        names=['category', 'structural_difficulty']
                    )))
    return overall, by_task, by_structure

## Individual variant visualisations

In [ ]:
for variant in VARIANTS:
    subset = task_df[task_df['variant'].eq(variant)]
    overall, by_task, by_structure = accuracy_tables(subset)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
    bars = axes[0].bar([variant], [overall], color=COLOURS[variant])
    label_bars(axes[0], bars); finish_rate_axis(axes[0], 'Overall success rate')
    bars = axes[1].bar(by_task.index, by_task.values, color=COLOURS[variant])
    label_bars(axes[1], bars); finish_rate_axis(axes[1], 'Success rate by task')
    labels = [f'{task}\n{difficulty.replace("_", " ")}'
              for task, difficulty in by_structure.index]
    bars = axes[2].bar(labels, by_structure.values, color=COLOURS[variant])
    label_bars(axes[2], bars); finish_rate_axis(axes[2], 'Success by task and structure')
    axes[2].tick_params(axis='x', rotation=35)
    fig.suptitle(f'{variant}: success analysis', fontsize=15)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{variant.lower().replace(" ", "_")}_success.png',
                dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
phase_rows = phase_summary.set_index('variant')
for variant in VARIANTS:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    plot_metric_set(axes[0], phase_rows, PRE, [variant], 'Phase 1: Preprocessing')
    plot_metric_set(axes[1], phase_rows, PLAN, [variant], 'Phase 2: Query planning')
    axes[2].axis('off'); axes[2].set_title('Phase 3: SQL generation')
    axes[2].text(.5, .52, 'Not independently\nscored', ha='center', fontsize=13)
    plot_metric_set(axes[3], phase_rows, POST, [variant], 'Phase 4: Post-processing')
    fig.suptitle(f'{variant}: phase-specific metrics', fontsize=15)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{variant.lower().replace(" ", "_")}_phases.png',
                dpi=160, bbox_inches='tight')
    plt.show()

## Pairwise comparisons

Each figure contains exactly one requested comparison.

In [ ]:
COMPARISONS = [('Variant 0', 'Variant 1'), ('Variant 1', 'Variant 3')]
for left, right in COMPARISONS:
    variants = [left, right]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    overall_values = [task_df.loc[task_df['variant'].eq(v), 'final_passed'].mean()
                      for v in variants]
    bars = axes[0].bar(variants, overall_values, color=[COLOURS[v] for v in variants])
    label_bars(axes[0], bars); finish_rate_axis(axes[0], 'Overall success rate')
    task_rates = (task_df[task_df['variant'].isin(variants)]
                  .groupby(['category', 'variant'])['final_passed'].mean().unstack())
    task_rates = task_rates.reindex(TASK_ORDER)[variants]
    task_rates.plot.bar(ax=axes[1], color=[COLOURS[v] for v in variants], width=.75)
    finish_rate_axis(axes[1], 'Success rate by task'); axes[1].tick_params(axis='x', rotation=0)
    structural = (task_df[task_df['variant'].isin(variants)]
                  .groupby(['category', 'structural_difficulty', 'variant'])['final_passed']
                  .mean().unstack())
    structural = structural.reindex(pd.MultiIndex.from_product(
        [TASK_ORDER, DIFFICULTY_ORDER], names=['category', 'structural_difficulty']))[variants]
    structural.index = [f'{a}\n{b.replace("_", " ")}' for a, b in structural.index]
    structural.plot.bar(ax=axes[2], color=[COLOURS[v] for v in variants], width=.78)
    finish_rate_axis(axes[2], 'Success by task and structure')
    axes[1].legend(title=''); axes[2].legend(title='')
    fig.suptitle(f'{left} vs {right}: success comparison', fontsize=15)
    plt.tight_layout()
    stem = f'{left.lower().replace(" ", "_")}_vs_{right.lower().replace(" ", "_")}'
    fig.savefig(OUTPUT_DIR / f'{stem}_success.png', dpi=160, bbox_inches='tight')
    plt.show()

In [ ]:
for left, right in COMPARISONS:
    variants = [left, right]
    fig, axes = plt.subplots(1, 4, figsize=(23, 5))
    plot_metric_set(axes[0], phase_rows, PRE, variants, 'Phase 1: Preprocessing')
    plot_metric_set(axes[1], phase_rows, PLAN, variants, 'Phase 2: Query planning')
    axes[2].axis('off'); axes[2].set_title('Phase 3: SQL generation')
    axes[2].text(.5, .52, 'No independent metric', ha='center', fontsize=13)
    plot_metric_set(axes[3], phase_rows, POST, variants, 'Phase 4: Post-processing')
    axes[0].legend(); axes[1].legend(); axes[3].legend()
    fig.suptitle(f'{left} vs {right}: phase-specific comparison', fontsize=15)
    plt.tight_layout()
    stem = f'{left.lower().replace(" ", "_")}_vs_{right.lower().replace(" ", "_")}'
    fig.savefig(OUTPUT_DIR / f'{stem}_phases.png', dpi=160, bbox_inches='tight')
    plt.show()

## Variant 1 and Variant 3 token usage

In [ ]:
token_rows = []
for variant in ['Variant 1', 'Variant 3']:
    subset = task_df[task_df['variant'].eq(variant)]
    token_rows.append({
        'variant': variant,
        'input_tokens': subset['metric_input_tokens'].sum(),
        'cached_input_tokens': subset['metric_cached_input_tokens'].sum(),
        'uncached_input_tokens': subset['metric_uncached_input_tokens'].sum(),
        'output_tokens': subset['metric_output_tokens'].sum(),
        'total_tokens': subset['metric_total_tokens'].sum(),
        'average_tokens_per_task': subset['metric_total_tokens'].mean(),
    })
token_summary = pd.DataFrame(token_rows)
display(token_summary.style.format({c: '{:,.0f}' for c in token_summary.columns if c != 'variant'}))
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(token_summary['variant'], token_summary['average_tokens_per_task'],
              color=[COLOURS[v] for v in token_summary['variant']])
ax.bar_label(bars, labels=[f'{v:,.0f}' for v in token_summary['average_tokens_per_task']], padding=3)
ax.set_title('Average tokens per task: Variant 1 vs Variant 3'); ax.set_ylabel('Tokens')
ax.grid(axis='y', alpha=.25); plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'variant_1_vs_variant_3_tokens.png', dpi=160, bbox_inches='tight')
plt.show()
token_summary.to_csv(OUTPUT_DIR / 'token_summary.csv', index=False)